(Frantar & Alistarh, ICML 2023) [«SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot»](https://arxiv.org/abs/2301.00774)

Классические подходы к прунингу — magnitude pruning, движение весов, методы второго порядка вроде Optimal Brain Surgeon — предполагают, что после удаления весов сеть можно дообучить. Для модели в 175 миллиардов параметров (GPT-3, OPT-175B) это нереалистично: одна эпоха обучения стоит сотни тысяч долларов

Возникает задача *one-shot* (или *post-training*) *pruning*: имея уже обученную модель и небольшую калибровочную выборку (несколько сотен предложений), за один проход получить разреженную модель без дообучения. До SparseGPT методы такого класса работали только для CNN; на больших трансформерах magnitude pruning при 50% разреженности уже разрушал качество.

### 1.2. Формулировка

SparseGPT, как и многие методы post-training compression (например, GPTQ для квантизации), сводит задачу к независимой обработке каждого линейного слоя.

Пусть слой задаётся матрицей весов $W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$. На калибровочных данных мы прогоняем модель и собираем входы слоя $X \in \mathbb{R}^{d_{\text{in}} \times N}$, где $N$ — общее число токенов (батч × длина последовательности). Выход исходного слоя: $Y = WX$.

Задача — найти разреженную матрицу $\widehat{W}$ (с заданной структурой разреженности) и, возможно, обновлённые значения оставшихся весов, минимизируя ошибку реконструкции выхода слоя:

$$
\min_{\widehat{W}} \; \|WX - \widehat{W}X\|_F^2 \quad \text{при ограничении на маску } M
$$

где $M \in \{0,1\}^{d_{\text{out}} \times d_{\text{in}}}$ — бинарная маска, а $\widehat{W} = \widehat{W} \odot M$.

Важно: критерий — не обнулить «неважные» веса, а минимизировать ошибку выхода слоя. Эти две вещи различаются, и именно отсюда растёт всё преимущество метода над magnitude pruning.

### 1.3. Почему задача всё ещё сложная

Если бы маска $M$ была фиксирована заранее, задача сводилась бы к обычной квадратичной оптимизации по оставшимся весам — решалась бы в замкнутом виде. Но маску тоже надо выбирать, и выбор оптимальной маски — NP-трудная комбинаторная задача даже для одной строки. Для слоя GPT-3 с $d_{\text{in}} = 12288$ перебор невозможен.

Поэтому SparseGPT — это эвристика, которая *одновременно* и выбирает маску, и обновляет веса. Она опирается на классический Optimal Brain Surgeon, поэтому начнём с него.

---

## Глава 2. OBS review

### 2.1. Постановка OBS

OBS (Hassibi & Stork, 1993) решает задачу: какой один вес удалить и как пересчитать оставшиеся, чтобы лосс изменился минимально? Раскладывая лосс в ряд Тейлора около обученной точки ($g \approx 0$):

$$
\delta \mathcal{L} \approx \tfrac{1}{2} \, \delta w^\top H \, \delta w
$$

Удаление веса $w_q$ — это ограничение $e_q^\top \delta w + w_q = 0$, где $e_q$ — единичный орт. Минимизация $\delta\mathcal{L}$ при этом ограничении через лагранжиан даёт две классические формулы OBS:

$$
\delta w = -\frac{w_q}{[H^{-1}]_{qq}} \, H^{-1} e_q, \qquad \delta\mathcal{L}_q = \frac{w_q^2}{2 \, [H^{-1}]_{qq}}
$$

То есть: важность («saliency») веса — это $w_q^2 / [H^{-1}]_{qq}$, а после удаления все оставшиеся веса нужно сдвинуть в направлении $q$-го столбца $H^{-1}$.

### 2.2. Гессиан для послойной задачи

В послойной формулировке (раздел 1.2) лосс — это квадратичная функция:

$$
\mathcal{L}(W) = \|WX - W_{\text{orig}}X\|_F^2
$$

Её гессиан по весам *одной строки* $W_{i,:}$ равен:

$$
H = 2 X X^\top \in \mathbb{R}^{d_{\text{in}} \times d_{\text{in}}}
$$

Ключевое наблюдение: гессиан *одинаков* для всех строк матрицы $W$. Это потому, что строки слоя действуют на один и тот же вход $X$ независимо. Лосс распадается в сумму по строкам:

$$
\|WX - \widehat{W}X\|_F^2 = \sum_{i=1}^{d_{\text{out}}} \|W_{i,:} X - \widehat{W}_{i,:} X\|_2^2
$$

Значит, мы можем обрабатывать каждую строку независимо, но переиспользовать $H^{-1}$. На больших моделях, где $d_{\text{out}}$ может быть в десятки тысяч, это даёт колоссальную экономию.

### 2.3. Почему наивный OBS не масштабируется

Стандартный OBS работает так: удалить один вес, пересчитать $H^{-1}$ (или хотя бы его релевантные элементы), повторить. Каждое обновление $H^{-1}$ при удалении веса — это формула Шермана-Моррисона, $O(d^2)$ операций. Удалить половину из $d$ весов в строке — $O(d^3)$. Для $d_{\text{in}} = 12288$ это уже на грани, а делать это $d_{\text{out}}$ раз (для каждой строки) — невозможно.

Более того, в классическом OBS порядок удаления зависит от текущего состояния $H^{-1}$, поэтому разные строки удаляли бы веса в разном порядке — и переиспользовать $H^{-1}$ между строками не получается.

SparseGPT решает обе проблемы одним приёмом

---

## Глава 3. Главный трюк SparseGPT: фиксированный порядок обработки

### 3.1. Идея

Зафиксируем порядок, в котором мы обрабатываем *столбцы* матрицы $W$: слева направо, от столбца 1 до столбца $d_{\text{in}}$. На каждом шаге $j$:

1. Для каждой строки решаем, оставить или обнулить вес в позиции $(i, j)$.
2. Веса в столбцах $> j$ (ещё не обработанные) обновляем, чтобы скомпенсировать сделанные изменения.
3. Веса в столбцах $< j$ (уже обработанные) больше не трогаем.

Это означает, что в момент обработки столбца $j$ актуальный гессиан — это нижне-правый блок исходного $H$, соответствующий ещё не зафиксированным весам. И этот блок одинаков для всех строк, потому что зависит только от $X$, а не от $W$.

Формально вводится последовательность «остаточных» обратных гессианов $H^{-1}_U$, где $U \subseteq \{1, \dots, d_{\text{in}}\}$ — множество ещё не обработанных индексов.

### 3.2. Эффективное обновление $H^{-1}$ через разложение Холецкого

Главная инженерная находка работы: вместо того чтобы заново считать обратные подматрицы или применять Шермана-Моррисона на каждом шаге, авторы используют верхне-треугольное разложение Холецкого $H^{-1} = L^\top L$ один раз в начале.

Утверждается следующее (это теорема 1 в статье): если последовательно удалять переменные в фиксированном порядке $1, 2, \dots$, то $j$-я строка матрицы $L$ — это в точности то, что нужно для обновления остаточного гессиана на шаге $j$.

Конкретно, обновление весов оставшихся столбцов после фиксации значения веса $w_{ij}$ становится:

$$
W_{i, j:} \leftarrow W_{i, j:} - \frac{w_{ij} - \widehat{w}_{ij}}{[H^{-1}]_{jj}} \cdot [H^{-1}]_{j, j:}
$$

где $\widehat{w}_{ij}$ — новое значение (либо 0, если обнулили, либо обновлённое значение из OBS), а $[H^{-1}]_{j, j:}$ — кусок $j$-й строки $H^{-1}$ от столбца $j$ и правее, который как раз и хранится в строке холецкого фактора.

Стоимость всего этого — одно разложение Холецкого размера $d_{\text{in}} \times d_{\text{in}}$ ($O(d_{\text{in}}^3)$, делается один раз для слоя) плюс $O(d_{\text{out}} \cdot d_{\text{in}}^2)$ на сами обновления. Это уже подъёмно для самых больших моделей.

### 3.3. Алгоритм выбора маски

Внутри каждого столбца $j$ мы решаем для каждой строки $i$ независимо: обнулять $w_{ij}$ или нет. Используется OBS-saliency:

$$
s_{ij} = \frac{w_{ij}^2}{[H^{-1}]_{jj}}
$$

Чем меньше $s_{ij}$, тем «безопаснее» обнулить вес. Знаменатель $[H^{-1}]_{jj}$ одинаков для всех строк в столбце, поэтому в рамках одного столбца порядок саlience-ов совпадает с порядком $w_{ij}^2$. Но между разными столбцами знаменатели различны — и именно поэтому SparseGPT отличается от magnitude pruning.

Маску можно выбирать разными стратегиями:

- *Глобально на слой*: выбрать $k$% весов с наименьшей saliency по всей матрице. Проблема: нарушается фиксированный порядок, поэтому делается приближённо — например, выбор маски заранее, до прохода.
- *Адаптивно по блокам*: матрица разбивается на блоки колонок ширины $B_s$ (например, 128). В пределах каждого блока выбирается локально $k$% слабейших весов, и решение принимается в начале блока — внутри блока порядок обработки уже сохраняется.

Адаптивный вариант — то, что используется в работе по умолчанию. Он даёт почти то же качество, что глобальный выбор, при сохранении эффективности.

### 3.4. Поддержка структурированных шаблонов разреженности

Метод естественно расширяется на N:M разреженность (например, 2:4): в каждой группе из $M$ подряд идущих весов нужно оставить ровно $N$. Алгоритм просто меняет правило выбора маски: внутри каждой группы из $M$ столбцов он выбирает $N$ позиций с наибольшей saliency и обнуляет остальные. Веса всё равно обновляются по той же OBS-формуле.

Это важно практически: 2:4 sparsity на NVIDIA Ampere/Hopper даёт честный 2x speedup без специальных sparse-ядер.

---

## Глава 4. Полный алгоритм

Алгоритм для одного линейного слоя $W$:

1. Прогнать калибровочные данные через модель, собрать входы слоя $X \in \mathbb{R}^{d_{\text{in}} \times N}$.
2. Посчитать $H = 2 X X^\top + \lambda I$ (демпфирование $\lambda$ нужно для численной устойчивости — обычно $\lambda = 0{,}01 \cdot \mathrm{tr}(H)/d_{\text{in}}$).
3. Один раз посчитать обратный гессиан и его разложение Холецкого: $H^{-1} = L^\top L$.
4. Для каждого блока столбцов $[j, j+B_s)$:
   - a. Посчитать saliency $s_{ij} = w_{ij}^2 / [H^{-1}]_{jj}$ для всех весов в блоке.
   - b. Выбрать маску для блока (например, $k$% наименьших saliency локально, или N:M в каждой группе).
   - c. Для каждого столбца $j'$ в блоке: для замаскированных позиций установить вес в 0, для оставшихся — оставить как есть, и применить OBS-обновление к весам в столбцах $\geq j'+1$:
   
$$
W_{i, j'+1:} \leftarrow W_{i, j'+1:} - \frac{w_{ij'} - \widehat{w}_{ij'}}{[H^{-1}]_{j'j'}} \cdot [H^{-1}]_{j', j'+1:}
$$

5. Перейти к следующему слою. Важно: входы $X$ для следующего слоя вычисляются на уже прунированной модели (последовательный режим), что компенсирует накопление ошибки.

Весь процесс — один проход вперёд по слоям модели. Нет ни одного шага градиентного обучения.

### Сложность

- Память: $O(d_{\text{in}}^2)$ для хранения $H$ и $L$.
- Время: $O(d_{\text{in}}^3 + d_{\text{out}} \cdot d_{\text{in}}^2)$ на слой.

Для OPT-175B весь прунинг занимает порядка 4 часов на одной A100. Это в тысячи раз быстрее, чем переобучить модель.

---

## Глава 5. Связи и интерпретации

### 5.1. SparseGPT vs magnitude pruning

Magnitude pruning ранжирует веса по $|w_{ij}|$. SparseGPT ранжирует по $w_{ij}^2 / [H^{-1}]_{jj}$ — то есть учитывает геометрию входных активаций. Если калибровочные данные на каком-то входном направлении имеют большую дисперсию, соответствующая диагональ $H$ велика, диагональ $H^{-1}$ мала, и веса на этом направлении считаются *более* важными.

### 5.2. SparseGPT vs Wanda

Wanda (Sun et al., 2023) — упрощение SparseGPT, появившееся через несколько месяцев. Она использует ту же интуицию, но берёт *диагональное* приближение гессиана: $[H^{-1}]_{jj} \approx 1 / \|X_{j,:}\|_2^2$. Скоринг становится:

$$
s_{ij}^{\text{Wanda}} = |w_{ij}| \cdot \|X_{j,:}\|_2
$$

Никаких обновлений весов, никакого разложения Холецкого. Результаты — близкие к SparseGPT на многих моделях, что является самостоятельным интересным наблюдением: значительная часть выигрыша SparseGPT приходится на правильное ранжирование, а не на пересчёт оставшихся весов.

### 5.3. SparseGPT vs GPTQ

GPTQ (тот же первый автор) — это аналогичный метод для квантизации: те же послойная задача, тот же приём с холецким разложением, тот же фиксированный порядок столбцов. Разница в том, что вместо «обнулить или оставить» решается «округлить к ближайшему уровню квантизации». Эти два метода можно комбинировать в один проход — получается одновременно квантизованная и разреженная модель.

### 5.4. Связь с OBC

OBC (Optimal Brain Compression, Frantar et al., 2022) — предыдущая работа тех же авторов, в которой и был заложен фундамент. SparseGPT — это в первую очередь масштабирование OBC до 100+ миллиардов параметров за счёт фиксированного порядка обработки и эффективного использования Холецкого. Концептуально это «OBC, который влезает в LLM».

---

## Глава 6. Эмпирические наблюдения из работы

- При 50% неструктурированной разреженности OPT-175B и BLOOM-176B почти не теряют в перплексии на WikiText-2 и тестах нулевого выстрела.
- Чем больше модель, тем легче её прунить — у моделей в 100B+ остаётся существенный запас перепараметризации. Маленькие модели (OPT-125M) ломаются уже при 50%.
- 2:4 sparsity даёт небольшую, но заметную просадку качества по сравнению с unstructured 50%. 4:8 — компромисс между ними.
- Семейство методов второго порядка чувствительно к шуму в калибровочной выборке; обычно используют 128–2048 последовательностей.
- При комбинировании с 4-битной квантизацией (SparseGPT + GPTQ) деградация качества почти аддитивна, но не катастрофична.

---

## Глава 7. Ограничения

- Метод послойный и жадный: он минимизирует ошибку реконструкции *выхода каждого слоя*, а не *итоговый лосс модели*. Ошибки могут накапливаться нелинейно через нелинейности.
- Зависимость от калибровочных данных. На сильно отличающемся распределении прунированная модель может вести себя хуже.
- Не решает задачу выбора маски *оптимально* — это эвристика. На очень высоких степенях разреженности (>70% для LLM) метод заметно деградирует.
- Для структурированного прунинга (удаление каналов, голов) SparseGPT неприменим — он работает только с неструктурированной или N:M разреженностью внутри матриц.

---

## Резюме

SparseGPT — это адаптация Optimal Brain Surgeon к задаче послойной реконструкции выхода в больших трансформерах. Главные идеи:

- Сведение прунинга всей модели к независимой обработке линейных слоёв.
- Фиксированный порядок обработки столбцов, позволяющий переиспользовать обратный гессиан между строками.
- Эффективное хранение и применение $H^{-1}$ через одно разложение Холецкого.
- Совместный выбор маски и пересчёт оставшихся весов в одном проходе, без дообучения.

Метод открыл целое направление — post-training compression для LLM — и стал базовой техникой, на которой строятся современные пайплайны сжатия больших моделей.